# PHASE 6.5.2 — CRISPR Essentialit

In [3]:
import pandas as pd

# load DepMap sample metadata
sample_info = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\sample_info.csv"
)

# sanity check
sample_info.shape, sample_info.columns


((1840, 29),
 Index(['DepMap_ID', 'cell_line_name', 'stripped_cell_line_name', 'CCLE_Name',
        'alias', 'COSMICID', 'sex', 'source', 'RRID', 'WTSI_Master_Cell_ID',
        'sample_collection_site', 'primary_or_metastasis', 'primary_disease',
        'Subtype', 'age', 'Sanger_Model_ID', 'depmap_public_comments',
        'lineage', 'lineage_subtype', 'lineage_sub_subtype',
        'lineage_molecular_subtype', 'default_growth_pattern',
        'model_manipulation', 'model_manipulation_details', 'patient_id',
        'parent_depmap_id', 'Cellosaurus_NCIt_disease', 'Cellosaurus_NCIt_id',
        'Cellosaurus_issues'],
       dtype='object'))

In [5]:
# keep only breast cancer cell lines
breast_cells = sample_info[
    sample_info["primary_disease"].str.contains("breast", case=False, na=False)
].copy()

# keep only required columns
breast_cells = breast_cells[
    [
        "DepMap_ID",
        "cell_line_name",
        "primary_disease",
        "lineage_molecular_subtype",
        "lineage_subtype",
    ]
]

breast_cells.shape, breast_cells.head()


((83, 5),
       DepMap_ID cell_line_name primary_disease lineage_molecular_subtype  \
 44   ACH-000288         BT-549   Breast Cancer                   basal_B   
 170  ACH-001388      SUM-102PT   Breast Cancer                     basal   
 171  ACH-001389    SUM-1315MO2   Breast Cancer                     basal   
 172  ACH-001393      SUM-190PT   Breast Cancer                   basal_A   
 173  ACH-001395       SUM-44PE   Breast Cancer                   luminal   
 
              lineage_subtype  
 44   breast_ductal_carcinoma  
 170         breast_carcinoma  
 171  breast_ductal_carcinoma  
 172         breast_carcinoma  
 173         breast_carcinoma  )

In [7]:
# quick subtype availability check
breast_cells["lineage_molecular_subtype"].value_counts(dropna=False), \
breast_cells["lineage_subtype"].value_counts(dropna=False)


(lineage_molecular_subtype
 NaN                 20
 luminal             17
 basal_A             16
 HER2_amp            13
 basal_B             11
 basal                4
 luminal_HER2_amp     2
 Name: count, dtype: int64,
 lineage_subtype
 breast_ductal_carcinoma    37
 breast_carcinoma           30
 NaN                        13
 breast_adenocarcinoma       3
 Name: count, dtype: int64)

In [9]:
### STEP 2 — Clean & standardize BC_subtype (single source of truth)


In [11]:
def map_bc_subtype(row):
    v = str(row["lineage_molecular_subtype"]).lower()
    
    if "luminal" in v and "her2" not in v:
        return "Luminal"
    if "her2" in v:
        return "HER2"
    if "basal" in v:
        return "TNBC"
    
    return None


breast_cells["BC_subtype"] = breast_cells.apply(map_bc_subtype, axis=1)

# drop unmapped
breast_cells = breast_cells.dropna(subset=["BC_subtype"])

breast_cells["BC_subtype"].value_counts(), breast_cells.shape


(BC_subtype
 TNBC       31
 Luminal    17
 HER2       15
 Name: count, dtype: int64,
 (63, 6))

In [13]:
### STEP 3 — Load CRISPR gene effect matrix (Chronos)


In [15]:
import pandas as pd

crispr = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\CRISPR_gene_effect.csv",
    index_col=0
)

crispr.shape, crispr.iloc[:5, :5]


((1086, 17386),
             A1BG (1)  A1CF (29974)   A2M (2)  A2ML1 (144568)  A3GALT2 (127550)
 DepMap_ID                                                                     
 ACH-000001 -0.134808      0.059764 -0.008665       -0.003572         -0.106211
 ACH-000004  0.081853     -0.056401 -0.106738       -0.014499          0.078209
 ACH-000005 -0.094196     -0.014598  0.100426        0.169103          0.032363
 ACH-000007 -0.011544     -0.123189  0.080692        0.061046         -0.013454
 ACH-000009 -0.050782     -0.037466  0.068885        0.090375          0.012634)

In [17]:
### STEP 4 — Align CRISPR with breast cancer subtypes (CRITICAL)


In [19]:
# subset CRISPR matrix to breast cancer cell lines
crispr_breast = crispr.loc[
    crispr.index.isin(breast_cells["DepMap_ID"])
]

crispr_breast.shape, crispr_breast.head()


((42, 17386),
             A1BG (1)  A1CF (29974)   A2M (2)  A2ML1 (144568)  \
 DepMap_ID                                                      
 ACH-000017  0.022619     -0.044618  0.022898        0.015679   
 ACH-000019  0.019367     -0.009001  0.123129        0.078343   
 ACH-000028 -0.189200     -0.078908  0.103560        0.128316   
 ACH-000097 -0.056046      0.031408  0.079729        0.136268   
 ACH-000111  0.080361     -0.108478 -0.013014        0.157375   
 
             A3GALT2 (127550)  A4GALT (53947)  A4GNT (51146)  AAAS (8086)  \
 DepMap_ID                                                                  
 ACH-000017         -0.295114       -0.118244       0.067409    -0.167049   
 ACH-000019         -0.048533        0.068221      -0.030091    -0.352269   
 ACH-000028         -0.077387       -0.010870      -0.092803    -0.380419   
 ACH-000097         -0.028177       -0.071391       0.055394    -0.335492   
 ACH-000111         -0.297995        0.068766      -0.002968    -0.

In [21]:
## STEP 5 — Split CRISPR by subtype (NEXT)


In [23]:
# split CRISPR by subtype
crispr_her2 = crispr_breast.loc[
    crispr_breast.index.isin(
        breast_cells.loc[breast_cells["BC_subtype"] == "HER2", "DepMap_ID"]
    )
]

crispr_tnbc = crispr_breast.loc[
    crispr_breast.index.isin(
        breast_cells.loc[breast_cells["BC_subtype"] == "TNBC", "DepMap_ID"]
    )
]

crispr_luminal = crispr_breast.loc[
    crispr_breast.index.isin(
        breast_cells.loc[breast_cells["BC_subtype"] == "Luminal", "DepMap_ID"]
    )
]

crispr_her2.shape, crispr_tnbc.shape, crispr_luminal.shape


((9, 17386), (24, 17386), (9, 17386))

In [25]:
## STEP 6 — Subtype-specific CRISPR essentiality (ONE clean cell)


In [33]:
# STEP 0 — load drug_targets (this is the missing object)

import pandas as pd

drug_targets = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv"
)

drug_targets.shape, drug_targets.head()


((522, 3),
   target_chembl_id    targets targets_ppi
 0       CHEMBL1778  ['IL2RA']   ['IL2RA']
 1       CHEMBL1782   ['FDPS']    ['FDPS']
 2       CHEMBL1783  ['VEGFA']   ['VEGFA']
 3       CHEMBL1785  ['EDNRB']   ['EDNRB']
 4       CHEMBL1786  ['IMPA1']   ['IMPA1'])

In [35]:
# convert targets_ppi list → long format (drug, gene)

import ast

drug_targets_long = (
    drug_targets
    .assign(
        targets_ppi=lambda df: df["targets_ppi"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    )
    .explode("targets_ppi")
    .rename(columns={
        "target_chembl_id": "drug",
        "targets_ppi": "gene"
    })[["drug", "gene"]]
)

drug_targets_long.shape, drug_targets_long.head()


((522, 2),
          drug   gene
 0  CHEMBL1778  IL2RA
 1  CHEMBL1782   FDPS
 2  CHEMBL1783  VEGFA
 3  CHEMBL1785  EDNRB
 4  CHEMBL1786  IMPA1)

In [37]:
## STEP 2 — prepare CRISPR matrix + breast cell lines (ONE cell)


In [39]:
# inputs assumed already loaded:
# 1) crispr : CRISPR_gene_effect.csv (index = DepMap_ID, columns = genes)
# 2) breast_cells : dataframe with DepMap_ID and BC_subtype
#    BC_subtype ∈ {"Luminal", "HER2", "TNBC"}

# sanity checks
print("CRISPR shape:", crispr.shape)
print("Breast cells by subtype:")
print(breast_cells["BC_subtype"].value_counts())

# keep only breast cancer cell lines present in CRISPR
breast_ids = set(breast_cells["DepMap_ID"])
crispr_breast = crispr.loc[crispr.index.intersection(breast_ids)]

crispr_breast.shape


CRISPR shape: (1086, 17386)
Breast cells by subtype:
BC_subtype
TNBC       31
Luminal    17
HER2       15
Name: count, dtype: int64


(42, 17386)

In [41]:
## STEP 3 — Compute subtype-specific CRISPR essentiality (CORE STEP)


In [45]:
import pandas as pd

def compute_crispr_by_subtype(subtype):
    # 1) select cell lines of this subtype
    cells = breast_cells.loc[
        breast_cells["BC_subtype"] == subtype, "DepMap_ID"
    ]

    # 2) subset CRISPR matrix
    crispr_sub = crispr.loc[crispr.index.intersection(cells)]

    records = []

    # 3) iterate drugs
    for drug, gdf in drug_targets_long.groupby("drug"):
        genes = list(set(gdf["gene"]) & set(crispr_sub.columns))
        if len(genes) == 0:
            continue

        score = crispr_sub[genes].mean().mean()

        records.append({
            "drug": drug,
            f"crispr_essentiality_{subtype.lower()}": score,
            f"n_targets_in_crispr_{subtype.lower()}": len(genes)
        })

    return pd.DataFrame(records)


In [47]:
## STEP 4 — Run CRISPR essentiality for EACH subtype 


In [51]:
# clean CRISPR column names: "TP53 (7157)" → "TP53"
crispr.columns = crispr.columns.str.split(" ").str[0]

# sanity
crispr.columns[:5]


Index(['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A3GALT2'], dtype='object')

In [55]:
def compute_crispr_by_subtype(subtype):
    cells = breast_cells.loc[
        breast_cells["BC_subtype"] == subtype, "DepMap_ID"
    ]

    crispr_sub = crispr.loc[crispr.index.intersection(cells)]

    records = []
    for drug, gdf in drug_network_genes.groupby("drug"):
        genes = list(set(gdf["gene"]) & set(crispr_sub.columns))
        if len(genes) == 0:
            continue

        score = crispr_sub[genes].mean().mean()

        records.append({
            "drug": drug,
            f"crispr_essentiality_{subtype.lower()}": score,
            f"n_crispr_genes_{subtype.lower()}": len(genes)
        })

    return pd.DataFrame(records)


In [57]:
## STEP 7 — CRISPR Essentiality (Luminal subtype ONLY)


# function to compute CRISPR essentiality by subtype
def compute_crispr_by_subtype(subtype):
    # select cell lines of this subtype
    cells = breast_cells.loc[
        breast_cells["BC_subtype"] == subtype, "DepMap_ID"
    ]
    
    crispr_sub = crispr.loc[crispr.index.intersection(cells)]
    
    records = []
    for drug, gdf in drug_targets_long.groupby("drug"):
        genes = list(set(gdf["gene"]) & set(crispr_sub.columns))
        if len(genes) == 0:
            continue
        
        score = crispr_sub[genes].mean().mean()
        records.append({
            "drug": drug,
            f"crispr_essentiality_{subtype.lower()}": score,
            f"n_targets_in_crispr_{subtype.lower()}": len(genes)
        })
    
    return pd.DataFrame(records)


# RUN for Luminal
crispr_luminal = compute_crispr_by_subtype("Luminal")

crispr_luminal.shape, crispr_luminal.head()


((508, 3),
          drug  crispr_essentiality_luminal  n_targets_in_crispr_luminal
 0  CHEMBL1778                     0.006238                            1
 1  CHEMBL1782                    -0.792907                            1
 2  CHEMBL1785                     0.049661                            1
 3  CHEMBL1786                     0.002927                            1
 4  CHEMBL1787                     0.060135                            1)

In [59]:

crispr_luminal.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_CRISPR_Essentiality_Luminal.csv",
    index=False
)


In [61]:
## Phase 6.5.2 — CRISPR Essentiality (HER2)

### Step: Compute drug–CRISPR essentiality for HER2 cell lines


In [63]:
# compute CRISPR essentiality for HER2 subtype

def compute_crispr_by_subtype(subtype_label, out_col_prefix):
    # select HER2 cell lines
    cells = breast_cells.loc[
        breast_cells["BC_subtype"] == subtype_label, "DepMap_ID"
    ]
    
    crispr_sub = crispr.loc[crispr.index.intersection(cells)]
    
    records = []
    for drug, gdf in drug_targets_long.groupby("drug"):
        genes = list(set(gdf["gene"]) & set(crispr_sub.columns))
        if len(genes) == 0:
            continue
        
        score = crispr_sub[genes].mean().mean()
        
        records.append({
            "drug": drug,
            f"crispr_essentiality_{out_col_prefix}": score,
            f"n_targets_in_crispr_{out_col_prefix}": len(genes)
        })
    
    return pd.DataFrame(records)


crispr_her2 = compute_crispr_by_subtype("HER2", "her2")

crispr_her2.shape, crispr_her2.head()


((508, 3),
          drug  crispr_essentiality_her2  n_targets_in_crispr_her2
 0  CHEMBL1778                 -0.034745                         1
 1  CHEMBL1782                 -0.679322                         1
 2  CHEMBL1785                  0.088629                         1
 3  CHEMBL1786                  0.009221                         1
 4  CHEMBL1787                  0.081892                         1)

In [65]:

crispr_her2.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_CRISPR_Essentiality_Her2.csv",
    index=False
)


In [67]:
## Phase 6.5.2 — CRISPR Essentiality (TNBC)


In [69]:
# compute CRISPR essentiality for TNBC subtype

crispr_tnbc = compute_crispr_by_subtype("TNBC", "tnbc")

crispr_tnbc.shape, crispr_tnbc.head()


((508, 3),
          drug  crispr_essentiality_tnbc  n_targets_in_crispr_tnbc
 0  CHEMBL1778                 -0.011116                         1
 1  CHEMBL1782                 -0.601927                         1
 2  CHEMBL1785                  0.043352                         1
 3  CHEMBL1786                 -0.017806                         1
 4  CHEMBL1787                  0.070106                         1)

In [71]:
crispr_tnbc.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_CRISPR_Essentiality_TNBC.csv",
    index=False
)
